# Confidence Intervals


## Learning Objectives

At the end of these exercises, you should be able to:

- Choose a confidence-interval method (z, t, or bootstrap) from the sample size and whether the population parameters are known.
- Construct a 95% confidence interval for a mean from sample data.
- Interpret a given confidence interval to answer a business question.
- Construct a bootstrap confidence interval on the Iris dataset and compare it to the z-formula.

If you need the worked method, see [`2_ci_guide.ipynb`](2_ci_guide.ipynb).


Implement the 4 steps that we learnt regarding distributions namely:

1. Understand the Data Type: Continuous or discrete.
2. Identify the Analytical Question: Determine which distribution fits your question.
3. Calculate Descriptive Statistics: Compute and visualize summary statistics.
4. Interpret the Graph and Descriptive Statistics: Analyze the data’s distribution and patterns.

To ultimately calculate the CI.

Keep in mind to check the following when selecting the method to calculate CI:

- Sample Size
- Population Parameters are known or unknown

---

## Case Study 1:

Scenario:
A tech company has introduced a new smartphone model. The battery life (in hours) of the smartphone is a key feature, and the company conducted a test on a sample of 50 units. The battery life data from this test is normally distributed. However, since this is a new product, the population mean and standard deviation of battery life are unknown.

Your task is to calculate the 95% confidence interval for the mean battery life of the smartphone.

Dataset: battery_life

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

battery = pd.read_csv("../../2_confidence_intervals/data/battery_life.csv")[
    "BatteryLife"
]
n = len(battery)
mean = battery.mean()
sem = battery.std(ddof=1) / np.sqrt(
    n
)  # population sigma unknown, estimate it from the sample
lo, hi = stats.t.interval(0.95, df=n - 1, loc=mean, scale=sem)
print(f"n = {n}, sample mean = {mean:.2f} hours")
print(f"95% t confidence interval for mean battery life: ({lo:.2f}, {hi:.2f}) hours")

**Explanation**

The population standard deviation is unknown, so we estimate it from the sample and use the t-distribution rather than the z. The interval gives the range where the true mean battery life plausibly sits. For large samples the t-interval is almost identical to the z-interval.

---

## Case Study 2:

Scenario:
A call center wants to estimate the average number of calls received per hour. A random sample of 40 hours is selected, and the number of calls received each hour is recorded. The number of calls follows a Poisson distribution, as it is a count of events within a fixed period. The population mean and variance are unknown.

Your task is to calculate the 95% confidence interval for the mean number of calls received per hour.

Dataset: calls_per_hour


In [ ]:
calls = pd.read_csv("../../2_confidence_intervals/data/calls_per_hour.csv")[
    "CallsPerHour"
]
n = len(calls)
mean = calls.mean()
sem = calls.std(ddof=1) / np.sqrt(n)
lo, hi = stats.t.interval(0.95, df=n - 1, loc=mean, scale=sem)
print(f"n = {n}, sample mean = {mean:.2f} calls/hour")
print(f"95% confidence interval for mean calls per hour: ({lo:.2f}, {hi:.2f})")

**Explanation**

The individual counts follow a Poisson distribution, but the question is about the mean number of calls, not a single hour. With n = 40 (above 30), the central limit theorem lets us treat the sample mean as approximately normal and use a t-interval, exactly as for the battery example.

---

## Case Study 3: Interpreting Confidence Interval for Customer Satisfaction
Scenario:
You work at a retail company, and your team conducted a survey to measure customer satisfaction with your online shopping platform. 

The survey asked customers to rate their overall satisfaction on a scale from 1 to 10, where 1 means "very dissatisfied" and 10 means "very satisfied."

After analyzing the responses from a random sample of 150 customers, the team calculated a 95% confidence interval for the mean satisfaction score, which is (7.2, 8.1).

```Task:
Your task is to interpret this confidence interval in the context of the survey results. Specifically, answer the following questions:

What does the confidence interval tell you about the average satisfaction score of all customers?

Can you confidently say that customers are generally satisfied with the online shopping platform? Why or why not?

How would you present these results to the company’s management team?
```

**Answer**

- What the interval tells you: we are 95% confident that the true mean satisfaction score of all customers lies between 7.2 and 8.1 on the 1 to 10 scale.
- Are customers generally satisfied? Yes. The entire interval sits well above the neutral midpoint of 5.5 and above 7, so even the low end points to broadly satisfied customers.
- How to present it to management: report that average satisfaction is about 7.2 to 8.1 out of 10, comfortably above neutral, so customers are generally satisfied with the platform.

---

## Case Study 4: Interpreting Confidence Interval for Product Defect Rate
Scenario:
You work at a manufacturing company. 

Your team has been monitoring the defect rate of a specific product. 

Over the past month, you randomly inspected 500 units and found that the defect rate was calculated along with a 95% confidence interval for the true defect rate, which is (3.5%, 6.5%).

```Task:
Interpret this confidence interval within the context of quality control. Address the following questions:

What does the confidence interval tell you about the true defect rate of the product?

Can you confidently say that the defect rate is below the company's acceptable threshold of 5%? Why or why not?

How would you explain these results to the production team? 
```

**Answer**

- What the interval tells you: we are 95% confident that the true product defect rate lies between 3.5% and 6.5%.
- Is the defect rate below the 5% threshold? We cannot say confidently. The interval straddles 5%, so the true rate could be either side of the limit. A confidence interval that crosses the threshold of interest is inconclusive on that question.
- How to explain it to the production team: the defect rate is estimated at 3.5% to 6.5%, and because that range crosses the 5% limit we cannot yet conclude we are under target. More units should be inspected to narrow the interval.

## Case Study 5: Iris Data set
For the Iris data set, calculate the 95% confidence intervals for the mean of the Length of the petal of Iris Setosa feature using bootstrapping. Create a notebook in which following questions shall be solved:

1. Initialize empty list called sample_means
2. Create a for loop that will loop 10,000 times
3. In each iteration using the pandas method .sample() sample 100 profits with replacement
4. Also in that same iteration take the mean of these 100 samples and then append it to the sample_means list
5. Using sns.displot plot the distribution of the sample_means list
6. Convert sample_means list to pandas Series
7. Using pandas .quantile() calculate the upper (0.975) and lower (0.025) limits of the confidence interval of the sample_means Series

### Bonus

8. Calculate using the z-value formula
9. Compare to the results of the bootstrapping method

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

In [ ]:
# Load the Iris dataset
iris = load_iris()
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)

In [ ]:
# Case Study 5: bootstrap CI for the mean Setosa petal length
petal = iris_df.loc[iris.target == 0, "petal length (cm)"]  # Setosa is target 0

np.random.seed(42)
sample_means = []
for _ in range(10000):
    resample = petal.sample(
        len(petal), replace=True
    )  # resample at the original sample size
    sample_means.append(resample.mean())
sample_means = pd.Series(sample_means)

sns.displot(sample_means)
plt.title("Bootstrap distribution of the mean Setosa petal length")
plt.show()

lower = sample_means.quantile(0.025)
upper = sample_means.quantile(0.975)
print(f"Bootstrap 95% CI for mean Setosa petal length: ({lower:.3f}, {upper:.3f}) cm")

# Bonus: the z-value formula, for comparison
mean = petal.mean()
sem = petal.std(ddof=1) / np.sqrt(len(petal))
z_lo, z_hi = stats.norm.interval(0.95, loc=mean, scale=sem)
print(f"z-formula 95% CI: ({z_lo:.3f}, {z_hi:.3f}) cm")

**Explanation**

Bootstrapping resamples the 50 Setosa petal lengths with replacement 10,000 times and takes the mean each time; the 2.5th and 97.5th percentiles of those means give the 95% interval without assuming normality. The z-formula interval is shown for comparison and comes out very close, because with n = 50 and a fairly symmetric variable the two methods agree. (The task text says "sample 100 profits", which is a copy-paste artifact: a bootstrap resamples at the original sample size, 50 here, of petal lengths.)

## Summary

In these exercises you:

- Selected the right confidence-interval method from the sample size and known or unknown parameters.
- Constructed confidence intervals for the battery-life and call-volume samples.
- Interpreted confidence intervals for customer satisfaction and product defect rate.
- Built a bootstrap confidence interval on the Iris data and compared it to the z-formula.

## References & Further Reading

- [**Confidence Intervals Guide**](2_ci_guide.ipynb): The worked method these exercises build on.
- [**scipy.stats reference**](https://docs.scipy.org/doc/scipy/reference/stats.html): The `norm` and `t` objects used for the formula-based intervals.
- [**NumPy random sampling**](https://numpy.org/doc/stable/reference/random/index.html): The resampling behind bootstrapping.
